# Running MoralBERT on VADER-informed polarity labels
This notebook creates a new ground truth dataset to compare against MoralBERT's outputs.

Demos for how to use VADER are here: https://github.com/cjhutto/vaderSentiment

## Preprocessing

Install VADER, sentiment labeling demonstration, and loading MFRC.

In [1]:
!pip install vaderSentiment

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 126.0/126.0 kB 3.3 MB/s eta 0:00:00


In [2]:
from vaderSentiment.vaderSentiment import SentimentIntensityAnalyzer

sentences = ["VADER is smart, handsome, and funny.",  # positive sentence example
             "VADER is smart, handsome, and funny!",  # punctuation emphasis handled correctly (sentiment intensity adjusted)
             "VADER is very smart, handsome, and funny.", # booster words handled correctly (sentiment intensity adjusted)
             "VADER is VERY SMART, handsome, and FUNNY.",  # emphasis for ALLCAPS handled
             "VADER is VERY SMART, handsome, and FUNNY!!!", # combination of signals - VADER appropriately adjusts intensity
             "VADER is VERY SMART, uber handsome, and FRIGGIN FUNNY!!!", # booster words & punctuation make this close to ceiling for score
             "VADER is not smart, handsome, nor funny.",  # negation sentence example
             "The book was good.",  # positive sentence
             "At least it isn't a horrible book.",  # negated negative sentence with contraction
             "The book was only kind of good.", # qualified positive sentence is handled correctly (intensity adjusted)
             "The plot was good, but the characters are uncompelling and the dialog is not great.", # mixed negation sentence
             "Today SUX!",  # negative slang with capitalization emphasis
             "Today only kinda sux! But I'll get by, lol", # mixed sentiment example with slang and constrastive conjunction "but"
             "Make sure you :) or :D today!",  # emoticons handled
             "Catch utf-8 emoji such as such as 💘 and 💋 and 😁",  # emojis handled
             "Not bad at all"  # Capitalized negation
             ]

analyzer = SentimentIntensityAnalyzer()
for sentence in sentences:
    vs = analyzer.polarity_scores(sentence)
    print("{:-<65} {}".format(sentence, str(vs)))

VADER is smart, handsome, and funny.----------------------------- {'neg': 0.0, 'neu': 0.254, 'pos': 0.746, 'compound': 0.8316}
VADER is smart, handsome, and funny!----------------------------- {'neg': 0.0, 'neu': 0.248, 'pos': 0.752, 'compound': 0.8439}
VADER is very smart, handsome, and funny.------------------------ {'neg': 0.0, 'neu': 0.299, 'pos': 0.701, 'compound': 0.8545}
VADER is VERY SMART, handsome, and FUNNY.------------------------ {'neg': 0.0, 'neu': 0.246, 'pos': 0.754, 'compound': 0.9227}
VADER is VERY SMART, handsome, and FUNNY!!!---------------------- {'neg': 0.0, 'neu': 0.233, 'pos': 0.767, 'compound': 0.9342}
VADER is VERY SMART, uber handsome, and FRIGGIN FUNNY!!!--------- {'neg': 0.0, 'neu': 0.294, 'pos': 0.706, 'compound': 0.9469}
VADER is not smart, handsome, nor funny.------------------------- {'neg': 0.646, 'neu': 0.354, 'pos': 0.0, 'compound': -0.7424}
The book was good.----------------------------------------------- {'neg': 0.0, 'neu': 0.508, 'pos': 0.492, 'co

In [3]:
from google.colab import drive
drive.mount("/content/drive")

Mounted at /content/drive


In [4]:
import pandas as pd

mfrc_df = pd.read_csv("/content/drive/MyDrive/datasets/final_mfrc_data.csv")
mfrc_df.head()

,text,subreddit,bucket,annotator,annotation,confidence
0,That particular part of the debate is especial...,europe,French politics,annotator03,Non-Moral,Confident
1,That particular part of the debate is especial...,europe,French politics,annotator01,Purity,Confident
2,That particular part of the debate is especial...,europe,French politics,annotator02,Thin Morality,Confident
3,"/r/france is pretty lively, with it's own ling...",europe,French politics,annotator03,Non-Moral,Confident
4,"/r/france is pretty lively, with it's own ling...",europe,French politics,annotator00,Non-Moral,Somewhat Confident


In [5]:
# drop irrelevant columns
mfrc_df = mfrc_df.drop(columns=["subreddit", "bucket", "annotator"])
mfrc_df.head()

,text,annotation,confidence
0,That particular part of the debate is especial...,Non-Moral,Confident
1,That particular part of the debate is especial...,Purity,Confident
2,That particular part of the debate is especial...,Thin Morality,Confident
3,"/r/france is pretty lively, with it's own ling...",Non-Moral,Confident
4,"/r/france is pretty lively, with it's own ling...",Non-Moral,Somewhat Confident


In [6]:
mfrc_df["confidence"].value_counts()

,count
confidence,
Confident,44737
Somewhat Confident,8920
Not Confident,7527


In [7]:
mfrc_df["annotation"].value_counts()

,count
annotation,
Non-Moral,30770
Thin Morality,11113
Care,3882
Authority,3111
Equality,2442
...,...
"Equality,Proportionality,Purity",1
"Equality,Care,Loyalty",1
"Equality,Care,Purity",1


In [6]:
# drop Non-Moral, drop Not Confident, drop Thin Morality
mfrc_df = mfrc_df[
    (mfrc_df["annotation"] != "Non-Moral") &
    (mfrc_df["annotation"] != "Thin Morality") &
    (mfrc_df["confidence"] != "Not Confident")
]
mfrc_df.head()

,text,annotation,confidence
1,That particular part of the debate is especial...,Purity,Confident
8,TBH Marion Le Pen would be better. Closet fasc...,Equality,Somewhat Confident
12,The Le Pen brand of conservatism and classical...,Authority,Somewhat Confident
21,"Hey, fuck you. Us leftists will never support ...","Loyalty,Equality",Confident
22,"Hey, fuck you. Us leftists will never support ...",Purity,Confident


In [7]:
# MFRC splits Fairness Cheating to Equality/Inequality and Proportionality/Disproportionality
# merge them back to their parent foundations

def normalize_foundations(annotation):
    foundations = annotation.split(',')
    foundations = [
        f.replace('Equality', 'Fairness')
            .replace('Proportionality', 'Fairness') for f in foundations]
    return ','.join(set(foundations))

mfrc_df["annotation"] = mfrc_df["annotation"].apply(normalize_foundations)
mfrc_df.head()

,text,annotation,confidence
1,That particular part of the debate is especial...,Purity,Confident
8,TBH Marion Le Pen would be better. Closet fasc...,Fairness,Somewhat Confident
12,The Le Pen brand of conservatism and classical...,Authority,Somewhat Confident
21,"Hey, fuck you. Us leftists will never support ...","Loyalty,Fairness",Confident
22,"Hey, fuck you. Us leftists will never support ...",Purity,Confident


 ### Generate sentiment scores with VADER

 How to handle cases where the text and confidence is the same but the annotations are different? I was thinking union; merge the individual rows into one and their annotations are pooled together.

In [8]:
def score_sentiment(text):
    vs = analyzer.polarity_scores(text)
    return vs["compound"]

mfrc_df["sentiment"] = mfrc_df["text"].apply(score_sentiment)
mfrc_df.head()

,text,annotation,confidence,sentiment
1,That particular part of the debate is especial...,Purity,Confident,-0.6596
8,TBH Marion Le Pen would be better. Closet fasc...,Fairness,Somewhat Confident,-0.6486
12,The Le Pen brand of conservatism and classical...,Authority,Somewhat Confident,0.9670
21,"Hey, fuck you. Us leftists will never support ...","Loyalty,Fairness",Confident,-0.8104
22,"Hey, fuck you. Us leftists will never support ...",Purity,Confident,-0.8104


### Generate polarity columns, then exude whether or not vice/virtue depending on sentiment score.

Let $x$ represent the input text.

Let $\alpha \in [-1, 1]$ represent the sentiment score observed from $x$.

Let $m \in M$ represent a moral foundation; where $m$ is a foundation and $M$ the set of all foundations.
- e.g. $m$ being Care/Harm moral axes

If $\alpha < 0$ then $x$ exudes vice poles of the annotations $m$ in the MFRC.

If $\alpha > 0$ then $x$ exudes virtue poles of the annotations $m$ in the MFRC.

If $\alpha = 0$ then $x$ exudes moral foundations $m$ but is polarity agnostic.
- can label the row `N` for neutral
  - this is something that warrants further scrutiny from either the literature or any of the profs we have access to.

In [9]:
# make new column `annotation_polarity` that takes
# the annotations and appends ".virtue" or ".vice"
# or just the label "N"

def label_polarity(annotations, sentiment):
  if sentiment == 0:
    return "N"

  new_annotations = []
  annotations = annotations.split(",")

  if sentiment > 0:
    for annotation in annotations:
      new_annotations.append(f"{annotation}.virtue")
  elif sentiment < 0:
    for annotation in annotations:
      new_annotations.append(f"{annotation}.vice")

  polarity_label = ",".join(new_annotations)

  return polarity_label

label_polarity("Loyalty,Equality", 0)

'N'

In [10]:
mfrc_df["polarity"] = mfrc_df.apply(
    lambda row: label_polarity(row["annotation"], row["sentiment"]),
    axis=1
)

mfrc_df.head()

,text,annotation,confidence,sentiment,polarity
1,That particular part of the debate is especial...,Purity,Confident,-0.6596,Purity.vice
8,TBH Marion Le Pen would be better. Closet fasc...,Fairness,Somewhat Confident,-0.6486,Fairness.vice
12,The Le Pen brand of conservatism and classical...,Authority,Somewhat Confident,0.9670,Authority.virtue
21,"Hey, fuck you. Us leftists will never support ...","Loyalty,Fairness",Confident,-0.8104,"Loyalty.vice,Fairness.vice"
22,"Hey, fuck you. Us leftists will never support ...",Purity,Confident,-0.8104,Purity.vice


### Shape `mfrc_df` into one hot encodings


In [11]:
# At this point the following columns can be dropped: `annotation`, `confidence`, and `sentiment`.
mfrc_df = mfrc_df.drop(columns=["annotation", "confidence", "sentiment"])
mfrc_df["polarity"] = mfrc_df["polarity"].str.lower()
mfrc_df.head()

,text,polarity
1,That particular part of the debate is especial...,purity.vice
8,TBH Marion Le Pen would be better. Closet fasc...,fairness.vice
12,The Le Pen brand of conservatism and classical...,authority.virtue
21,"Hey, fuck you. Us leftists will never support ...","loyalty.vice,fairness.vice"
22,"Hey, fuck you. Us leftists will never support ...",purity.vice


In [12]:
# generate blank columns for each polarity
labels = [
    # "text", "polarity",
    "care", "harm",
    "fairness", "cheating",
    "loyalty", "betrayal",
    "authority", "subversion",
    "purity", "degradation",
]

for label in labels:
  mfrc_df[label] = 0

mfrc_df.head()

,text,polarity,care,harm,fairness,cheating,loyalty,betrayal,authority,subversion,purity,degradation
1,That particular part of the debate is especial...,purity.vice,0,0,0,0,0,0,0,0,0,0
8,TBH Marion Le Pen would be better. Closet fasc...,fairness.vice,0,0,0,0,0,0,0,0,0,0
12,The Le Pen brand of conservatism and classical...,authority.virtue,0,0,0,0,0,0,0,0,0,0
21,"Hey, fuck you. Us leftists will never support ...","loyalty.vice,fairness.vice",0,0,0,0,0,0,0,0,0,0
22,"Hey, fuck you. Us leftists will never support ...",purity.vice,0,0,0,0,0,0,0,0,0,0


In [13]:
# one hot encode

POLARITY_MAP = {
    "care.virtue":       "care",
    "fairness.virtue":   "fairness",
    "loyalty.virtue":    "loyalty",
    "authority.virtue":  "authority",
    "purity.virtue":     "purity",

    "care.vice":         "harm",
    "fairness.vice":     "cheating",
    "loyalty.vice":      "betrayal",
    "authority.vice":    "subversion",
    "purity.vice":       "degradation",
}

ENCODING_COLS = ["care", "harm", "fairness", "cheating", "loyalty",
                 "betrayal", "authority", "subversion", "purity", "degradation"]

def encode_polarity(df):
    for idx, row in df.iterrows():
        polarity = str(row["polarity"]).strip()

        if polarity == "N":
            df.loc[idx, ENCODING_COLS] = "N"
        else:
            labels = polarity.split(",")
            for label in labels:
                label = label.strip()
                if label in POLARITY_MAP:
                    col = POLARITY_MAP[label]
                    df.at[idx, col] = 1
    return df

mfrc_df = encode_polarity(mfrc_df)
mfrc_df.head()

,text,polarity,care,harm,fairness,cheating,loyalty,betrayal,authority,subversion,purity,degradation
1,That particular part of the debate is especial...,purity.vice,0,0,0,0,0,0,0,0,0,1
8,TBH Marion Le Pen would be better. Closet fasc...,fairness.vice,0,0,0,1,0,0,0,0,0,0
12,The Le Pen brand of conservatism and classical...,authority.virtue,0,0,0,0,0,0,1,0,0,0
21,"Hey, fuck you. Us leftists will never support ...","loyalty.vice,fairness.vice",0,0,0,1,0,1,0,0,0,0
22,"Hey, fuck you. Us leftists will never support ...",purity.vice,0,0,0,0,0,0,0,0,0,1


In [16]:
# saving current OHE to Drive

# mfrc_df = mfrc_df.drop(columns=["polarity"])
# mfrc_df.to_csv("/content/drive/MyDrive/datasets/MFRC_VADER_labels.csv")

## Running MoralBERT inference

This needs a GPU runtime.

In [17]:
with open("/content/drive/MyDrive/keys/hfpat-colab.txt") as f:
  HF_TOKEN = f.read().strip()

from huggingface_hub import login
login(token=HF_TOKEN)

In [18]:
import numpy as np
import torch
import torch.nn as nn
from huggingface_hub import PyTorchModelHubMixin
from transformers import AutoModel, AutoTokenizer
import torch.nn.functional as F

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
bert_model = AutoModel.from_pretrained("bert-base-uncased").to(device)
tokenizer = AutoTokenizer.from_pretrained("bert-base-uncased")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertModel LOAD REPORT from: bert-base-uncased
Key                                        | Status     |  | 
-------------------------------------------+------------+--+-
cls.predictions.transform.LayerNorm.weight | UNEXPECTED |  | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED |  | 
cls.predictions.bias                       | UNEXPECTED |  | 
cls.predictions.transform.dense.weight     | UNEXPECTED |  | 
cls.predictions.transform.dense.bias       | UNEXPECTED |  | 
cls.seq_relationship.weight                | UNEXPECTED |  | 
cls.seq_relationship.bias                  | UNEXPECTED |  | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.


tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

In [19]:
class MyModel(
    nn.Module,
    PyTorchModelHubMixin,
    # optionally, you can add metadata which gets pushed to the model card
    # repo_url="your-repo-url",
    pipeline_tag="text-classification",
    license="mit",
):
    def __init__(self, bert_model, moral_label=2):

        super(MyModel, self).__init__()
        self.bert = bert_model
        bert_dim = 768
        self.invariant_trans = nn.Linear(768, 768)
        self.moral_classification = nn.Sequential(nn.Linear(768,768),
                                                      nn.ReLU(),
                                                      nn.Linear(768, moral_label))

    def forward(self, input_ids, token_type_ids, attention_mask):
        pooled_output = self.bert(input_ids,
                                token_type_ids = token_type_ids,
                                attention_mask = attention_mask).last_hidden_state[:,0,:]


        pooled_output = self.invariant_trans(pooled_output)


        logits = self.moral_classification(pooled_output)

        return logits

In [20]:
def preprocessing(input_text, tokenizer):
    return tokenizer(
                        input_text,
                        add_special_tokens = True,
                        max_length = 150,
                        padding = 'max_length',
                        return_attention_mask = True,
                        return_token_type_ids = True,  # Add this line
                        return_tensors = 'pt',
                        truncation=True
                   )

In [21]:
mft_values = ["care", "harm", "fairness", "cheating", "loyalty", "betrayal",
              "authority", "subversion", "purity", "degradation"]

models = {}
for mft in mft_values:
    repo_name = f"vjosap/moralBERT-predict-{mft}-in-text"
    models[mft] = MyModel.from_pretrained(repo_name, bert_model=bert_model).to(device)


config.json:   0%|          | 0.00/22.0 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/443M [00:00<?, ?B/s]

config.json:   0%|          | 0.00/22.0 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/443M [00:00<?, ?B/s]

config.json:   0%|          | 0.00/22.0 [00:00<?, ?B/s]

...

model.safetensors:   0%|          | 0.00/443M [00:00<?, ?B/s]

config.json:   0%|          | 0.00/22.0 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/443M [00:00<?, ?B/s]

config.json:   0%|          | 0.00/22.0 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/443M [00:00<?, ?B/s]

In [22]:
def get_batch_scores(sentences, mft, batch_size=1024):
    model = models[mft]
    model.eval()
    all_scores = []

    for i in range(0, len(sentences), batch_size):
        batch = list(sentences[i:i+batch_size])
        encodeds = tokenizer(batch, add_special_tokens=True, max_length=150,
                             padding='max_length', truncation=True,
                             return_attention_mask=True, return_token_type_ids=True,
                             return_tensors='pt')
        encodeds = {k: v.to(device) for k, v in encodeds.items()}

        with torch.no_grad():
            output = models[mft](**encodeds)
            scores = F.softmax(output, dim=1)[:, 1].tolist()
        all_scores.extend(scores)

    return all_scores


In [23]:
import tqdm
input_texts = list(mfrc_df["text"])
results = {"text": list(input_texts)}
for mft in tqdm.tqdm(mft_values, desc="Processing MFT models"):
    results[mft] = get_batch_scores(input_texts, mft)

preds = pd.DataFrame(results)

Processing MFT models: 100%|██████████| 10/10 [04:37<00:00, 27.79s/it]


In [26]:
preds.head()
# preds.to_csv("/content/drive/MyDrive/datasets/MoralBERT-raws-VADER.csv")

,text,care,harm,fairness,cheating,loyalty,betrayal,authority,subversion,purity,degradation
0,That particular part of the debate is especial...,0.349088,0.435736,0.369921,0.347554,0.347338,0.261093,0.320770,0.275342,0.481949,0.064982
1,TBH Marion Le Pen would be better. Closet fasc...,0.375630,0.642975,0.268419,0.513547,0.404092,0.304502,0.440248,0.234550,0.446407,0.286639
2,The Le Pen brand of conservatism and classical...,0.377618,0.330923,0.430411,0.383453,0.664201,0.574285,0.349016,0.162718,0.452946,0.007505
3,"Hey, fuck you. Us leftists will never support ...",0.305138,0.637172,0.326031,0.407409,0.432484,0.296277,0.368498,0.240756,0.417694,0.175725
4,"Hey, fuck you. Us leftists will never support ...",0.305138,0.637172,0.326031,0.407409,0.432484,0.296277,0.368498,0.240756,0.417694,0.175725


## Compute F1

Threshold tuning and what not.

In [25]:
preds = pd.read_csv("/content/drive/MyDrive/datasets/MoralBERT-raws-VADER.csv")
preds = preds.drop(columns=["Unnamed: 0"])
preds.head()

,text,care,harm,fairness,cheating,loyalty,betrayal,authority,subversion,purity,degradation
0,That particular part of the debate is especial...,0.349088,0.435736,0.369921,0.347554,0.347338,0.261093,0.320770,0.275342,0.481949,0.064982
1,TBH Marion Le Pen would be better. Closet fasc...,0.375630,0.642975,0.268419,0.513547,0.404092,0.304502,0.440248,0.234550,0.446407,0.286639
2,The Le Pen brand of conservatism and classical...,0.377618,0.330923,0.430411,0.383453,0.664201,0.574285,0.349016,0.162718,0.452946,0.007505
3,"Hey, fuck you. Us leftists will never support ...",0.305138,0.637172,0.326031,0.407409,0.432484,0.296277,0.368498,0.240756,0.417694,0.175725
4,"Hey, fuck you. Us leftists will never support ...",0.305138,0.637172,0.326031,0.407409,0.432484,0.296277,0.368498,0.240756,0.417694,0.175725


In [27]:
gt = mfrc_df.copy()
gt = gt.drop(columns=["polarity"])
gt.head()

,text,care,harm,fairness,cheating,loyalty,betrayal,authority,subversion,purity,degradation
1,That particular part of the debate is especial...,0,0,0,0,0,0,0,0,0,1
8,TBH Marion Le Pen would be better. Closet fasc...,0,0,0,1,0,0,0,0,0,0
12,The Le Pen brand of conservatism and classical...,0,0,0,0,0,0,1,0,0,0
21,"Hey, fuck you. Us leftists will never support ...",0,0,0,1,0,1,0,0,0,0
22,"Hey, fuck you. Us leftists will never support ...",0,0,0,0,0,0,0,0,0,1


In [28]:
import numpy as np
from sklearn.metrics import f1_score

labels = gt.drop(columns=["text"]).columns.tolist()

y_true = gt[labels].values
probs  = preds[labels].values

In [29]:
# single global threshold across polarities
thresholds = np.arange(0.05, 0.95, 0.05)

results = []

for t in thresholds:
    y_pred = (probs > t).astype(int)
    macro = f1_score(y_true, y_pred, average="macro")
    micro = f1_score(y_true, y_pred, average="micro")
    results.append((t, macro, micro))

pd.DataFrame(results, columns=["threshold", "macro_f1", "micro_f1"])

,threshold,macro_f1,micro_f1
0,0.05,0.247278,0.244654
1,0.10,0.249662,0.244614
2,0.15,0.251712,0.246602
3,0.20,0.249890,0.247468
4,0.25,0.244405,0.247625
5,0.30,0.244165,0.250859
6,0.35,0.247318,0.255250
7,0.40,0.246128,0.256570
8,0.45,0.230301,0.243047
9,0.50,0.182203,0.200375


In [30]:
# tuning thresholds per polarity
best_thresholds = {}

for i, label in enumerate(labels):
    best_f1 = 0
    best_t = 0.5

    for t in np.arange(0.05, 0.95, 0.05):
        pred = (probs[:, i] > t).astype(int)
        f1 = f1_score(y_true[:, i], pred)

        if f1 > best_f1:
            best_f1 = f1
            best_t = t

    best_thresholds[label] = best_t

best_thresholds

{'care': np.float64(0.4),
 'harm': np.float64(0.4),
 'fairness': np.float64(0.45),
 'cheating': np.float64(0.4),
 'loyalty': np.float64(0.6500000000000001),
 'betrayal': np.float64(0.25),
 'authority': np.float64(0.35000000000000003),
 'subversion': np.float64(0.15000000000000002),
 'purity': np.float64(0.5),
 'degradation': np.float64(0.3)}

In [31]:
y_pred = np.zeros_like(probs)

for i, label in enumerate(labels):
    t = best_thresholds[label]
    y_pred[:, i] = (probs[:, i] > t).astype(int)

print("Macro F1:", f1_score(y_true, y_pred, average="macro"))
print("Micro F1:", f1_score(y_true, y_pred, average="micro"))

Macro F1: 0.2806224634472058
Micro F1: 0.28406193165962607
